## 1. Imports & Load Raw Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os.path as path

In [2]:
df = pd.read_csv('data/AirQualityUCI.csv',sep=';')
df = df.drop(columns=['Unnamed: 15','Unnamed: 16'])
y_target = 'PT08.S1(CO)'
df.head()

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,10/03/2004,18.00.00,"2,6",1360.0,150.0,"11,9",1046.0,166.0,1056.0,113.0,1692.0,1268.0,"13,6","48,9","0,7578"
1,10/03/2004,19.00.00,2,1292.0,112.0,"9,4",955.0,103.0,1174.0,92.0,1559.0,972.0,"13,3","47,7","0,7255"
2,10/03/2004,20.00.00,"2,2",1402.0,88.0,"9,0",939.0,131.0,1140.0,114.0,1555.0,1074.0,"11,9","54,0","0,7502"
3,10/03/2004,21.00.00,"2,2",1376.0,80.0,"9,2",948.0,172.0,1092.0,122.0,1584.0,1203.0,"11,0","60,0","0,7867"
4,10/03/2004,22.00.00,"1,6",1272.0,51.0,"6,5",836.0,131.0,1205.0,116.0,1490.0,1110.0,"11,2","59,6","0,7888"


## 2. Convert Numeric String To Numerict type Float

In [3]:
feature_string = df.select_dtypes(include='object').columns.to_list()
kolom_tanggal = [col for col in feature_string if 'date' in col.lower() or 'time' in col.lower()]
kolom_object_lainnya = [col for col in feature_string if col not in kolom_tanggal]

for col in  kolom_object_lainnya:
    df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9471 entries, 0 to 9470
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           9357 non-null   object 
 1   Time           9357 non-null   object 
 2   CO(GT)         9357 non-null   float64
 3   PT08.S1(CO)    9357 non-null   float64
 4   NMHC(GT)       9357 non-null   float64
 5   C6H6(GT)       9357 non-null   float64
 6   PT08.S2(NMHC)  9357 non-null   float64
 7   NOx(GT)        9357 non-null   float64
 8   PT08.S3(NOx)   9357 non-null   float64
 9   NO2(GT)        9357 non-null   float64
 10  PT08.S4(NO2)   9357 non-null   float64
 11  PT08.S5(O3)    9357 non-null   float64
 12  T              9357 non-null   float64
 13  RH             9357 non-null   float64
 14  AH             9357 non-null   float64
dtypes: float64(13), object(2)
memory usage: 1.1+ MB


## 3. Handle Missing / Invalid Values

In [4]:
missing_feature = df.columns[df.isnull().any()].to_list()
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 1710
Duplicates: 113
Missing Feature:
['Date', 'Time', 'CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH']


In [5]:
for col in missing_feature:
    missing_percentage = (df[col].isnull().sum() / len(df)) * 100
    if missing_percentage > 5.0 :
        df = df.drop(columns=col)
    else:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            skewness = df[col].skew()
            if abs(skewness) < 0.5:
                df[col] = df[col].fillna(df[col].mean()).round(2)
            else:
                df[col] = df[col].fillna(df[col].median()).round(2)

print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 0
Missing Feature:
['Date', 'Time', 'CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH']


## 4. Handling Duplicated

In [6]:
df = df.drop_duplicates()

jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah data duplikat: {jumlah_duplikat}")

Jumlah data duplikat: 0


## 5. Analisis & Handling Outliers

In [7]:
feature_numerik = df.select_dtypes(include=np.number).columns
Q1 = df[feature_numerik].quantile(0.25)
Q3 = df[feature_numerik].quantile(0.75)
IQR = Q3-Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sebelum Dihapus: {outliers.shape[0]}")

#delete outliers
df = df.loc[((df[feature_numerik] >= lower_bound) & (df[feature_numerik] <= upper_bound)).all(axis=1)]
outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sesudah Dihapus: {outliers.shape[0]}")


Jumlah Outlier Sebelum Dihapus: 3895
Jumlah Outlier Sesudah Dihapus: 0


## 6. Feature Engineering

In [8]:
df[kolom_tanggal[1]] = df[kolom_tanggal[1]].astype(str).str.replace('.',':',regex=False)
datetime_string = df[kolom_tanggal[0]] + ' ' + df[kolom_tanggal[1]]
df['datetime'] = pd.to_datetime(datetime_string,format='%d/%m/%Y %H:%M:%S', errors='coerce')
df = df.drop(columns=['Date', 'Time'])

df['NO2_NOx_Ratio'] = df["NO2(GT)"] / (df['NOx(GT)'] + 1e-5)
df['T_RH_Interaction'] = df['T'] * df['RH']

df['Hour'] = df['datetime'].dt.hour
df['Month'] = df['datetime'].dt.month
df['Is_Weekend'] = df['datetime'].dt.dayofweek.isin([5, 6]).astype(int)

## 5.Save Cleaned Dataset

In [9]:
file_path = 'data/AirQualityUCI_Cleaning.csv'

if not path.exists(file_path):
    df.to_csv(file_path, index=False)
    print('File belum ada. Berhasil menyimpan dataset baru!')
else:
    print('File sudah ada. Proses penyimpanan CSV dilewati (skip)')

df.head()

File belum ada. Berhasil menyimpan dataset baru!


,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,datetime,NO2_NOx_Ratio,T_RH_Interaction,Hour,Month,Is_Weekend
184,4.5,1617.0,-200.0,21.3,1333.0,349.0,686.0,150.0,2010.0,1819.0,17.8,40.5,0.82,2004-03-18 10:00:00,0.429799,720.90,10,3,0
185,2.8,1473.0,-200.0,14.3,1127.0,224.0,831.0,152.0,1752.0,1568.0,20.8,34.4,0.84,2004-03-18 11:00:00,0.678571,715.52,11,3,0
186,2.2,1379.0,-200.0,12.5,1068.0,171.0,899.0,139.0,1663.0,1374.0,23.8,28.2,0.82,2004-03-18 12:00:00,0.812865,671.16,12,3,0
187,2.2,1385.0,-200.0,12.2,1056.0,149.0,891.0,133.0,1648.0,1268.0,24.2,28.7,0.85,2004-03-18 13:00:00,0.892617,694.54,13,3,0
188,2.3,1379.0,-200.0,13.1,1087.0,137.0,901.0,126.0,1660.0,1144.0,25.2,24.9,0.78,2004-03-18 14:00:00,0.919708,627.48,14,3,0
